In [ ]:
import pandas as pd
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

class StrokePrediction(BaseModel):
    reasoning: str = Field(description="Step-by-step clinical reasoning explaining the patient's risk.")
    probability: float = Field(description="Estimated probability of suffering a stroke (0.0 to 1.0).")
    prediction: int = Field(description="Final binary prediction: 1 (high stroke risk) or 0 (low risk).")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
def serialize_patient(row: pd.Series) -> str:
    """Convert a dataset row into a natural-language patient profile."""

    return f"""
    Patient Profile:
    - Age: {row['age']} years
    - Gender: {row['gender']}
    - BMI (Body Mass Index): {row['bmi']}
    - Average glucose level: {row['avg_glucose_level']} mg/dL
    - Hypertension: {'Yes' if row['hypertension'] == 1 else 'No'}
    - Heart Disease: {'Yes' if row['heart_disease'] == 1 else 'No'}
    - Marital status (Ever married): {row['ever_married']}
    - Work type: {row['work_type']}
    - Residence type: {row['Residence_type']}
    - Smoking status: {row['smoking_status']}
    """

In [ ]:
# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0.0,api_key="")  # Replace

# Force structured output
structured_llm = llm.with_structured_output(StrokePrediction)

# Create the prompt template
prompt = PromptTemplate.from_template(
    """You are an expert medical analyst. Your task is to assess the risk of the following patient suffering a stroke.

    Carefully analyze their health metrics, lifestyle, and demographic data.
    Consider the interactions between age, hypertension, glucose levels, and BMI.

    {patient_data}
    """
)

# Build the execution chain
prediction_chain = prompt | structured_llm

In [ ]:
# Cargar y preparar una muestra de los datos
df = pd.read_csv(r"C:\Users\i34005\OneDrive - Wood Mackenzie Limited\AI on healthcare\hw6\data\healthcare-dataset-stroke-data.csv")

# Imputación básica rápida para el ejemplo
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# Tomar un paciente de prueba
sample_patient = df.iloc[0]
patient_text = serialize_patient(sample_patient)

# Ejecutar la cadena
result = prediction_chain.invoke({"patient_data": patient_text})

print(f"Predicción Binaria: {result.prediction}")
print(f"Probabilidad: {result.probability}")
print(f"Razonamiento Clínico:\n{result.reasoning}")